# Module 03: Layers - Building Blocks of Neural Networks

In [33]:
import numpy as np
rng = np.random.default_rng(7)

import sys

# Add project root to path
sys.path.append(r"D:\DS\Data Science\TinyTorch\projects\my-tinytorch-systems")

from tinytorch.foundation.tensor import Tensor
from tinytorch.foundation.activations import Sigmoid, ReLU

# Constants for weight initialization
# Note: True Xavier/Glorot uses sqrt(2/(fan_in+fan_out)), but we use the simpler
# LeCun-style sqrt(1/fan_in) for pedagogical clarity. Both achieve stable gradients.
INIT_SCALE_FACTOR = 1.0  # LeCun-style initialization: sqrt(1/fan_in)
HE_SCALE_FACTOR = 2.0  # He initialization uses sqrt(2/fan_in) for ReLU

# Constants for dropout
DROPOUT_MIN_PROB = 0.0  # Minimum dropout probability (no dropout)
DROPOUT_MAX_PROB = 1.0  # Maximum dropout probability (drop everything)


# Layer Base Class - Foundation for All Layers

In [34]:
class Layer:
    """
    Base class for all neural network layers.
    """

    def forward(self, x):
        """
        Forward pass through the layer.

        Args:
            x: Input tensor

        Returns:
            Output tensor after transformation
        """
        
        raise NotImplementedError(
            f"forward() not implemented in {self.__class__.__name__}\n"
            f"  ❌ The Layer base class requires subclasses to implement forward()\n"
            f"  💡 forward() defines how input data is transformed by this layer\n"
            f"  🔧 Add this method to your class:\n"
            f"     def forward(self, x):\n"
            f"         # Your transformation logic here\n"
            f"         return transformed_x"
        )

    def __call__(self, x, *args, **kwargs):
        """Allow layer to be called like a function."""
        return self.forward(x, *args, **kwargs)

    def parameters(self):
        """
        Return list of trainable parameters.

        Returns:
            List of Tensor objects (weights and biases)
        """
        return []  # Base class has no parameters

    def __repr__(self):
        """String representation of the layer."""
        return f"{self.__class__.__name__}()"

# Linear Layer - The Foundation of Neural Networks

In [35]:
class Linear(Layer):
    """
    Linear (fully connected) layer: y = xW + b
    """

    def __init__(self, in_features, out_features, bias=True):
        """
        Initialize linear layer with proper weight initialization.
        """

        self.in_features = in_features
        self.out_features = out_features

        # LeCun-style initialization for stable gradients
        scale = np.sqrt(INIT_SCALE_FACTOR / in_features)
        weight_data = rng.standard_normal((in_features, out_features)) * scale
        self.weight = Tensor(weight_data)

        # Initialize bias to zeros or None
        if bias:
            bias_data = np.zeros(out_features)
            self.bias = Tensor(bias_data)
        else:
            self.bias = None
       

    def forward(self, x):
        """
        Forward pass through linear layer.
        """
      
        # Linear transformation: y = xW
        output = x.matmul(self.weight)

        # Add bias if present
        if self.bias is not None:
            output = output + self.bias

        return output

    def parameters(self):
        """
        Return list of trainable parameters.
        """
        
        params = [self.weight]
        if self.bias is not None:
            params.append(self.bias)
        return params
        

    def __repr__(self):
        """String representation for debugging."""
        bias_str = f", bias={self.bias is not None}"
        return f"Linear(in_features={self.in_features}, out_features={self.out_features}{bias_str})"

### Unit Test: Linear Layer

This test validates our Linear layer implementation works correctly.

In [36]:
def test_unit_linear_layer():
    """ Test Linear layer implementation."""
    print(" Unit Test: Linear Layer...")

    # Test layer creation
    layer = Linear(784, 256)
    assert layer.in_features == 784
    assert layer.out_features == 256
    assert layer.weight.shape == (784, 256)
    assert layer.bias.shape == (256,)

    # Test LeCun-style initialization (weights should be reasonably scaled)
    weight_std = np.std(layer.weight.data)
    expected_std = np.sqrt(INIT_SCALE_FACTOR / 784)
    assert 0.5 * expected_std < weight_std < 2.0 * expected_std, f"Weight std {weight_std} not close to expected {expected_std}"

    # Test bias initialization (should be zeros)
    assert np.allclose(layer.bias.data, 0), "Bias should be initialized to zeros"

    # Test forward pass
    x = Tensor(rng.standard_normal((32, 784)))  # Batch of 32 samples
    y = layer.forward(x)
    assert y.shape == (32, 256), f"Expected shape (32, 256), got {y.shape}"

    # Test no bias option
    layer_no_bias = Linear(10, 5, bias=False)
    assert layer_no_bias.bias is None
    params = layer_no_bias.parameters()
    assert len(params) == 1  # Only weight, no bias

    # Test parameters method
    params = layer.parameters()
    assert len(params) == 2  # Weight and bias
    assert params[0] is layer.weight
    assert params[1] is layer.bias

    print("✅ Linear layer works correctly!")

if __name__ == "__main__":
    test_unit_linear_layer()

 Unit Test: Linear Layer...
✅ Linear layer works correctly!


### Edge Case Tests: Linear Layer
Additional tests for edge cases and error handling.

In [37]:
def test_unit_edge_cases_linear():
    """ Test Linear layer edge cases."""
    print(" Edge Case Tests: Linear Layer...")

    layer = Linear(10, 5)

    # Test single sample (should handle 2D input)
    x_2d = Tensor(rng.standard_normal((1, 10)))
    y = layer.forward(x_2d)
    assert y.shape == (1, 5), "Should handle single sample"

    # Test zero batch size (edge case)
    x_empty = Tensor(rng.standard_normal((0, 10)))
    y_empty = layer.forward(x_empty)
    assert y_empty.shape == (0, 5), "Should handle empty batch"

    # Test numerical stability with large weights
    layer_large = Linear(10, 5)
    layer_large.weight.data = np.ones((10, 5)) * 100  # Large but not extreme
    x = Tensor(np.ones((1, 10)))
    y = layer_large.forward(x)
    assert not np.any(np.isnan(y.data)), "Should not produce NaN with large weights"
    assert not np.any(np.isinf(y.data)), "Should not produce Inf with large weights"

    # Test with no bias
    layer_no_bias = Linear(10, 5, bias=False)
    x = Tensor(rng.standard_normal((4, 10)))
    y = layer_no_bias.forward(x)
    assert y.shape == (4, 5), "Should work without bias"

    print("✅ Edge cases handled correctly!")

if __name__ == "__main__":
    test_unit_edge_cases_linear()

 Edge Case Tests: Linear Layer...
✅ Edge cases handled correctly!


### Parameter Collection Tests: Linear Layer
Tests to ensure Linear layer parameters can be collected for optimization.

In [38]:
def test_unit_parameter_collection_linear():
    """Test Linear layer parameter collection."""
    print("Parameter Collection Test: Linear Layer...")

    layer = Linear(10, 5)

    # Verify parameter collection works
    params = layer.parameters()
    assert len(params) == 2, "Should return 2 parameters (weight and bias)"
    assert params[0].shape == (10, 5), "First param should be weight"
    assert params[1].shape == (5,), "Second param should be bias"

    # Test layer without bias
    layer_no_bias = Linear(10, 5, bias=False)
    params_no_bias = layer_no_bias.parameters()
    assert len(params_no_bias) == 1, "Should return 1 parameter (weight only)"

    print("✅ Parameter collection works correctly!")

if __name__ == "__main__":
    test_unit_parameter_collection_linear()

Parameter Collection Test: Linear Layer...
✅ Parameter collection works correctly!


### Dropout Layer - Preventing Overfitting
Dropout is a regularization technique that randomly "turns off" neurons during training. This forces the network to not rely too heavily on any single neuron, making it more robust and generalizable.

In [39]:
class Dropout(Layer):
    """
    Dropout layer for regularization.

    During training: randomly zeros elements with probability p, scales survivors by 1/(1-p)
    During inference: passes input through unchanged

    This prevents overfitting by forcing the network to not rely on specific neurons.
    """

    def __init__(self, p=0.5):
        """
        Initialize dropout layer.
        """
        
        if not DROPOUT_MIN_PROB <= p <= DROPOUT_MAX_PROB:
            raise ValueError(
                f"Invalid dropout probability: {p}\n"
                f"  ❌ p must be between {DROPOUT_MIN_PROB} and {DROPOUT_MAX_PROB}\n"
                f"  💡 p is the probability of DROPPING a neuron (not keeping it!)\n"
                f"     p=0.0 means keep all neurons (no dropout)\n"
                f"     p=0.5 means drop 50% of neurons randomly\n"
                f"     p=1.0 means drop all neurons (zero output)\n"
                f"  🔧 Common values: Dropout(0.1) for light, Dropout(0.3) for moderate, Dropout(0.5) for aggressive"
            )
        self.p = p
       

    def _should_apply_dropout(self, training):
        """
        Determine whether dropout should be applied.

        Dropout is a training-time technique. During inference the full
        network is used, so dropout is skipped. It is also skipped when p=0
        (no neurons are dropped) since the result would be the identity.

        """
       
        return training and self.p > DROPOUT_MIN_PROB
        

    def _generate_dropout_mask(self, shape):
        """
        Generate a random dropout mask with inverted scaling.
        """
        
        keep_prob = 1.0 - self.p
        binary_mask = (rng.random(shape) < keep_prob).astype(np.float32)
        scale = 1.0 / keep_prob
        return Tensor(binary_mask * scale)
       

    def forward(self, x, training=True):
        """
        Forward pass through dropout layer.

        Composes the two helpers: first decide whether dropout applies,
        then generate and apply the mask if it does.
        """
        
        if not self._should_apply_dropout(training):
            return x

        if self.p == DROPOUT_MAX_PROB:
            return Tensor(np.zeros_like(x.data))

        mask = self._generate_dropout_mask(x.data.shape)
        return x * mask
       

    def __call__(self, x, training=True):
        """Allows the layer to be called like a function."""
        return self.forward(x, training)

    def parameters(self):
        """Dropout has no parameters."""
        return []

    def __repr__(self):
        return f"Dropout(p={self.p})"

## Unit Test: Dropout Decision Logic

In [40]:
def test_unit_should_apply_dropout():
    """ Test _should_apply_dropout decision logic."""
    print(" Unit Test: Dropout Decision Logic...")

    # Standard dropout (p=0.5) in training mode should apply
    d = Dropout(0.5)
    assert d._should_apply_dropout(training=True) is True, \
        "Dropout(0.5) should apply during training"

    # Same dropout in inference mode should NOT apply
    assert d._should_apply_dropout(training=False) is False, \
        "Dropout should not apply during inference"

    # Zero dropout (p=0) should never apply, even in training
    d_zero = Dropout(0.0)
    assert d_zero._should_apply_dropout(training=True) is False, \
        "Dropout(0.0) should never apply (no neurons to drop)"

    # Full dropout (p=1.0) in training mode should apply
    d_full = Dropout(1.0)
    assert d_full._should_apply_dropout(training=True) is True, \
        "Dropout(1.0) should apply during training"

    # Full dropout in inference mode should NOT apply
    assert d_full._should_apply_dropout(training=False) is False, \
        "Even Dropout(1.0) should not apply during inference"

    print("✅ Dropout decision logic works correctly!")

if __name__ == "__main__":
    test_unit_should_apply_dropout()

 Unit Test: Dropout Decision Logic...
✅ Dropout decision logic works correctly!


###  Unit Test: Dropout Mask Generation

In [41]:
def test_unit_generate_dropout_mask():
    """ Test _generate_dropout_mask output properties."""
    print("Unit Test: Dropout Mask Generation...")

    d = Dropout(0.5)
    rng = np.random.default_rng(7)
    mask = d._generate_dropout_mask((1000,))

    # Shape must match the requested shape
    assert mask.shape == (1000,), f"Expected shape (1000,), got {mask.shape}"

    # Every element must be either 0.0 or 2.0 (= 1/(1-0.5))
    unique_vals = set(np.unique(mask.data))
    assert unique_vals <= {0.0, 2.0}, \
        f"Mask values should be {{0.0, 2.0}}, got {unique_vals}"

    # Statistically, about 50% should survive (3-sigma tolerance)
    non_zero = np.count_nonzero(mask.data)
    std_err = np.sqrt(1000 * 0.5 * 0.5)
    assert 500 - 3 * std_err < non_zero < 500 + 3 * std_err, \
        f"Expected ~500 survivors, got {non_zero}"

    # Test with different dropout probability
    d2 = Dropout(0.3)
    rng = np.random.default_rng(7)
    mask2 = d2._generate_dropout_mask((2000,))

    # Values should be 0.0 or 1/(1-0.3) ≈ 1.4286
    expected_scale = 1.0 / 0.7
    non_zero_vals = mask2.data[mask2.data != 0.0]
    assert np.allclose(non_zero_vals, expected_scale), \
        f"Surviving values should be {expected_scale:.4f}, got {np.unique(non_zero_vals)}"

    # About 70% should survive for p=0.3
    survival_rate = np.count_nonzero(mask2.data) / 2000
    assert 0.60 < survival_rate < 0.80, \
        f"Expected ~70% survival for p=0.3, got {survival_rate:.1%}"

    print("✅ Dropout mask generation works correctly!")

if __name__ == "__main__":
    test_unit_generate_dropout_mask()

Unit Test: Dropout Mask Generation...
✅ Dropout mask generation works correctly!


## Sequential - Layer Container for Composition

In [42]:
class Sequential:
    """
    Container that chains layers together sequentially.
    """

    def __init__(self, *layers):
        """Initialize with layers to chain together."""
        # Accept both Sequential(layer1, layer2) and Sequential([layer1, layer2])
        if len(layers) == 1 and isinstance(layers[0], (list, tuple)):
            self.layers = list(layers[0])
        else:
            self.layers = list(layers)

    def forward(self, x, training=True):
        """Forward pass through all layers sequentially.

        Passes training=True/False to layers that support it (e.g. Dropout),
        and falls back to a plain forward(x) call for layers that don't.
        This lets you switch between training and eval mode with one flag:

            output = model.forward(x, training=False)   # eval: Dropout disabled
            output = model.forward(x, training=True)    # train: Dropout active
        """
        for layer in self.layers:
            try:
                x = layer.forward(x, training=training)
            except TypeError:
                x = layer.forward(x)
        return x

    def __call__(self, x, training=True):
        """Allow model to be called like a function."""
        return self.forward(x, training=training)

    def parameters(self):
        """Collect all parameters from all layers."""
        params = []
        for layer in self.layers:
            params.extend(layer.parameters())
        return params

    def __repr__(self):
        layer_reprs = ", ".join(repr(layer) for layer in self.layers)
        return f"Sequential({layer_reprs})"

## Unit Test: Dropout Layer
This test validates our Dropout layer implementation works correctly.

In [43]:
def test_unit_dropout_layer():
    """ Test Dropout layer implementation."""
    print(" Unit Test: Dropout Layer...")

    # Test dropout creation
    dropout = Dropout(0.5)
    assert dropout.p == 0.5

    # Test inference mode (should pass through unchanged)
    x = Tensor([1, 2, 3, 4])
    y_inference = dropout.forward(x, training=False)
    assert np.array_equal(x.data, y_inference.data), "Inference should pass through unchanged"

    # Test training mode with zero dropout (should pass through unchanged)
    dropout_zero = Dropout(0.0)
    y_zero = dropout_zero.forward(x, training=True)
    assert np.array_equal(x.data, y_zero.data), "Zero dropout should pass through unchanged"

    # Test training mode with full dropout (should zero everything)
    dropout_full = Dropout(1.0)
    y_full = dropout_full.forward(x, training=True)
    assert np.allclose(y_full.data, 0), "Full dropout should zero everything"

    # Test training mode with partial dropout
    # Note: This is probabilistic, so we test statistical properties
    rng = np.random.default_rng(7)  # For reproducible test
    x_large = Tensor(np.ones((1000,)))  # Large tensor for statistical significance
    y_train = dropout.forward(x_large, training=True)

    # Count non-zero elements (approximately 50% should survive)
    non_zero_count = np.count_nonzero(y_train.data)
    expected = 500
    # Use 3-sigma bounds: std = sqrt(n*p*(1-p)) = sqrt(1000*0.5*0.5) ≈ 15.8
    std_error = np.sqrt(1000 * 0.5 * 0.5)
    lower_bound = expected - 3 * std_error  # ≈ 453
    upper_bound = expected + 3 * std_error  # ≈ 547
    assert lower_bound < non_zero_count < upper_bound, \
        f"Expected {expected}±{3*std_error:.0f} survivors, got {non_zero_count}"

    # Test scaling (surviving elements should be scaled by 1/(1-p) = 2.0)
    surviving_values = y_train.data[y_train.data != 0]
    expected_value = 2.0  # 1.0 / (1 - 0.5)
    assert np.allclose(surviving_values, expected_value), f"Surviving values should be {expected_value}"

    # Test no parameters
    params = dropout.parameters()
    assert len(params) == 0, "Dropout should have no parameters"

    # Test invalid probability
    try:
        Dropout(-0.1)
        assert False, "Should raise ValueError for negative probability"
    except ValueError:
        pass

    try:
        Dropout(1.1)
        assert False, "Should raise ValueError for probability > 1"
    except ValueError:
        pass

    print("✅ Dropout layer works correctly!")

if __name__ == "__main__":
    test_unit_dropout_layer()

 Unit Test: Dropout Layer...
✅ Dropout layer works correctly!


## Integration: Bringing It Together
Now that we've built both layer types, let's see how they work together to create a complete neural network architecture. We'll manually compose a realistic 3-layer MLP for MNIST digit classification.

## Systems Analysis: Memory and Performance
Now let's analyze the systems characteristics of our layer implementations. Understanding memory usage and computational complexity helps us build efficient neural networks.

In [44]:
def analyze_layer_memory():
    """ Analyze memory usage patterns in layer operations."""
    print(" Analyzing Layer Memory Usage...")

    # Test different layer sizes
    layer_configs = [
        (784, 256),   # MNIST → hidden
        (256, 256),   # Hidden → hidden
        (256, 10),    # Hidden → output
        (2048, 2048), # Large hidden
    ]

    print("\nLinear Layer Memory Analysis:")
    print("Configuration → Weight Memory → Bias Memory → Total Memory")

    for in_feat, out_feat in layer_configs:
        # Calculate memory usage
        weight_memory = in_feat * out_feat * 4  # 4 bytes per float32
        bias_memory = out_feat * 4
        total_memory = weight_memory + bias_memory

        print(f"({in_feat:4d}, {out_feat:4d}) → {weight_memory/1024:7.1f} KB → {bias_memory/1024:6.1f} KB → {total_memory/1024:7.1f} KB")

    # Analyze multi-layer memory scaling
    print("\n💡 Multi-layer Model Memory Scaling:")
    hidden_sizes = [128, 256, 512, 1024, 2048]

    for hidden_size in hidden_sizes:
        # 3-layer MLP: 784 → hidden → hidden/2 → 10
        layer1_params = 784 * hidden_size + hidden_size
        layer2_params = hidden_size * (hidden_size // 2) + (hidden_size // 2)
        layer3_params = (hidden_size // 2) * 10 + 10

        total_params = layer1_params + layer2_params + layer3_params
        memory_mb = total_params * 4 / (1024 * 1024)

        print(f"Hidden={hidden_size:4d}: {total_params:7,} params = {memory_mb:5.1f} MB")

# Run the analysis
if __name__ == "__main__":
    analyze_layer_memory()

 Analyzing Layer Memory Usage...

Linear Layer Memory Analysis:
Configuration → Weight Memory → Bias Memory → Total Memory
( 784,  256) →   784.0 KB →    1.0 KB →   785.0 KB
( 256,  256) →   256.0 KB →    1.0 KB →   257.0 KB
( 256,   10) →    10.0 KB →    0.0 KB →    10.0 KB
(2048, 2048) → 16384.0 KB →    8.0 KB → 16392.0 KB

💡 Multi-layer Model Memory Scaling:
Hidden= 128: 109,386 params =   0.4 MB
Hidden= 256: 235,146 params =   0.9 MB
Hidden= 512: 535,818 params =   2.0 MB
Hidden=1024: 1,333,770 params =   5.1 MB
Hidden=2048: 3,716,106 params =  14.2 MB


In [45]:
def analyze_layer_performance():
    """Analyze computational complexity of layer operations."""
    import time

    print(" Analyzing Layer Computational Complexity...")

    # Test forward pass FLOPs
    batch_sizes = [1, 32, 128, 512]
    layer = Linear(784, 256)

    print("\nLinear Layer MACs Analysis:")
    print("Batch Size → Matrix Multiply MACs → Bias Add MACs → Total MACs")
    print("Note: FLOPs = 2 × MACs (one multiply + one add per MAC)")

    for batch_size in batch_sizes:
        # Matrix multiplication: (batch, in) @ (in, out) = batch * in * out MACs
        matmul_flops = batch_size * 784 * 256
        # Bias addition: batch * out MACs
        bias_flops = batch_size * 256
        total_flops = matmul_flops + bias_flops

        print(f"{batch_size:10d} → {matmul_flops:15,} → {bias_flops:13,} → {total_flops:11,}")

    # Add timing measurements
    print("\nLinear Layer Timing Analysis:")
    print("Batch Size → Time (ms) → Throughput (samples/sec)")

    for batch_size in batch_sizes:
        x = Tensor(rng.standard_normal((batch_size, 784)))

        # Warm up
        for _ in range(10):
            _ = layer.forward(x)

        # Time multiple iterations
        iterations = 100
        start = time.perf_counter()
        for _ in range(iterations):
            _ = layer.forward(x)
        elapsed = time.perf_counter() - start

        time_per_forward = (elapsed / iterations) * 1000  # Convert to ms
        throughput = (batch_size * iterations) / elapsed

        print(f"{batch_size:10d} → {time_per_forward:8.3f} ms → {throughput:12,.0f} samples/sec")

    print("\n💡 Key Insights:")
    print("🚀 Linear layer complexity: O(batch_size × in_features × out_features)")
    print("🚀 Memory grows linearly with batch size, quadratically with layer width")
    print("🚀 Dropout adds minimal computational overhead (element-wise operations)")
    print("🚀 Larger batches amortize overhead, improving throughput efficiency")

# Run the analysis
if __name__ == "__main__":
    analyze_layer_performance()

 Analyzing Layer Computational Complexity...

Linear Layer MACs Analysis:
Batch Size → Matrix Multiply MACs → Bias Add MACs → Total MACs
Note: FLOPs = 2 × MACs (one multiply + one add per MAC)
         1 →         200,704 →           256 →     200,960
        32 →       6,422,528 →         8,192 →   6,430,720
       128 →      25,690,112 →        32,768 →  25,722,880
       512 →     102,760,448 →       131,072 → 102,891,520

Linear Layer Timing Analysis:
Batch Size → Time (ms) → Throughput (samples/sec)
         1 →    0.475 ms →        2,107 samples/sec
        32 →   11.980 ms →        2,671 samples/sec
       128 →   48.948 ms →        2,615 samples/sec
       512 →  190.055 ms →        2,694 samples/sec

💡 Key Insights:
🚀 Linear layer complexity: O(batch_size × in_features × out_features)
🚀 Memory grows linearly with batch size, quadratically with layer width
🚀 Dropout adds minimal computational overhead (element-wise operations)
🚀 Larger batches amortize overhead, improving throu

## Module Integration Test
Final validation that everything works together correctly.

In [46]:
def test_module():
    """ Module Test: Complete Integration
    """
    print(" RUNNING MODULE INTEGRATION TEST")
    print("=" * 50)

    # Run all unit tests
    print("Running unit tests...")
    test_unit_linear_layer()
    test_unit_edge_cases_linear()
    test_unit_parameter_collection_linear()
    test_unit_should_apply_dropout()
    test_unit_generate_dropout_mask()
    test_unit_dropout_layer()

    print("\nRunning integration scenarios...")

    # Test realistic neural network construction with manual composition
    print(" Integration Test: Multi-layer Network...")

    # Use ReLU imported from package at module level
    ReLU_class = ReLU

    # Build individual layers for manual composition
    layer1 = Linear(784, 128)
    activation1 = ReLU_class()
    dropout1 = Dropout(0.5)
    layer2 = Linear(128, 64)
    activation2 = ReLU_class()
    dropout2 = Dropout(0.3)
    layer3 = Linear(64, 10)

    # Test end-to-end forward pass with manual composition
    batch_size = 16
    x = Tensor(rng.standard_normal((batch_size, 784)))

    # Manual forward pass
    x = layer1.forward(x)
    x = activation1.forward(x)
    x = dropout1.forward(x)
    x = layer2.forward(x)
    x = activation2.forward(x)
    x = dropout2.forward(x)
    output = layer3.forward(x)

    assert output.shape == (batch_size, 10), f"Expected output shape ({batch_size}, 10), got {output.shape}"

    # Test parameter counting from individual layers
    all_params = layer1.parameters() + layer2.parameters() + layer3.parameters()
    expected_params = 6  # 3 weights + 3 biases from 3 Linear layers
    assert len(all_params) == expected_params, f"Expected {expected_params} parameters, got {len(all_params)}"

    # Test individual layer functionality
    test_x = Tensor(rng.standard_normal((4, 784)))
    # Test dropout in training vs inference
    dropout_test = Dropout(0.5)
    train_output = dropout_test.forward(test_x, training=True)
    infer_output = dropout_test.forward(test_x, training=False)
    assert np.array_equal(test_x.data, infer_output.data), "Inference mode should pass through unchanged"

    print("✅ Multi-layer network integration works!")

    print("\n" + "=" * 50)
    print(" ALL TESTS PASSED!.")

## Layers Transform Shapes
What you built: Linear layers that transform data from one dimension to another.

In [47]:
def demo_layers():
    """🎯 See how layers transform shapes."""
    print("🎯 AHA MOMENT: Layers Transform Shapes")
    print("=" * 45)

    # Create a layer that transforms 784 → 10 (like MNIST)
    layer = Linear(784, 10)

    # Simulate a batch of 32 flattened images
    batch = Tensor(rng.standard_normal((32, 784)))

    # Forward pass
    output = layer(batch)

    print(f"Input shape:  {batch.shape}  ← 32 images, 784 pixels each")
    print(f"Output shape: {output.shape}  ← 32 images, 10 classes each")
    print(f"Parameters:   {784 * 10 + 10:,} (weights + biases)")

    print("\n✨ Your layer transforms images to class predictions!")

In [48]:
if __name__ == "__main__":
    test_module()
    print("\n")
    demo_layers()

 RUNNING MODULE INTEGRATION TEST
Running unit tests...
 Unit Test: Linear Layer...
✅ Linear layer works correctly!
 Edge Case Tests: Linear Layer...
✅ Edge cases handled correctly!
Parameter Collection Test: Linear Layer...
✅ Parameter collection works correctly!
 Unit Test: Dropout Decision Logic...
✅ Dropout decision logic works correctly!
Unit Test: Dropout Mask Generation...
✅ Dropout mask generation works correctly!
 Unit Test: Dropout Layer...
✅ Dropout layer works correctly!

Running integration scenarios...
 Integration Test: Multi-layer Network...
✅ Multi-layer network integration works!

 ALL TESTS PASSED!.


🎯 AHA MOMENT: Layers Transform Shapes
Input shape:  (32, 784)  ← 32 images, 784 pixels each
Output shape: (32, 10)  ← 32 images, 10 classes each
Parameters:   7,850 (weights + biases)

✨ Your layer transforms images to class predictions!
